In [ ]:
# Cell Growth Analysis: Measuring the Area of an E. coli Colony
# Install the required Python packages
#
# numpy is used for numerical calculations and image arrays.
# matplotlib is used for displaying images and making plots.
# pillow is used to open TIFF image files.
# gdown is used to download the shared image-data folder from Google Drive.
#
# In Google Colab, run this cell first.

%pip install -q numpy matplotlib pillow gdown


In [ ]:
# Import the Python tools we will use

# NumPy is used for numerical calculations and for storing images as arrays.
import numpy as np

# Matplotlib is used to display images and create plots.
import matplotlib.pyplot as plt

# Pillow is used to open TIFF image files.
from PIL import Image

# Path makes it easier to work with folder names and file names.
from pathlib import Path

# gdown allows Colab to download a shared Google Drive folder.
import gdown

# These tools are useful for displaying output inside a Jupyter/Colab notebook.
from IPython.display import display, clear_output

# time can be used later to pause between displayed image frames.
import time


In [ ]:
# Download the colony-growth image data from the shared Google Drive folder.
#
# The students do not need to mount Google Drive or change a Windows file path.
# Running this cell downloads the TIFF images into the temporary Colab workspace.

# Shared Google Drive folder containing the colony_growth_XX.tif image files.
folder_url = "https://drive.google.com/drive/folders/16nHWbift1Ye549ab-CelR5vPjlgQgJvt?usp=sharing"

# This is the folder where the images will be stored inside the Colab session.
DATA_DIR = Path("/content/ColonyGrowthData")

# Download the folder only if the TIFF images are not already present.
# This prevents unnecessary downloading if the cell is run more than once.
if not list(DATA_DIR.glob("*.tif")):
    gdown.download_folder(
        url=folder_url,
        output=str(DATA_DIR),
        quiet=False
    )

# Print the folder location used by Python.
print("Data folder:", DATA_DIR)

# Check whether the folder exists after downloading.
print("Folder exists:", DATA_DIR.exists())

# Count the TIFF images found in the folder.
image_files = sorted(DATA_DIR.glob("*.tif"))
print("Number of TIFF images found:", len(image_files))

# Show the first few file names so we can confirm that the download worked.
print("\nExample image files:")
for file in image_files[:5]:
    print(file.name)


In [ ]:
# Load one image: frame 26
# DATA_DIR contains the folder location.
# The / operator joins that folder with the image file name.
# The result is the complete path to colony_growth_26.tif.
image_path = DATA_DIR / "colony_growth_26.tif"

# Image.open() opens the TIFF image file using Pillow.
pil_image = Image.open(image_path)

# np.array() converts the opened image into a NumPy array.
# For a grayscale image, each array element represents the brightness
# (pixel intensity) of one pixel.
image = np.array(pil_image)

# Create a new figure that is 6 inches wide and 6 inches high.
plt.figure(figsize=(6, 6))

# Display the numerical image array as an image.
# cmap="gray" means low values appear dark and high values appear bright.
plt.imshow(image, cmap="gray")

# Add a title above the image.
plt.title("E. coli Colony - Frame 26")

# Hide the x- and y-axis ticks because they are not needed for this image view.
plt.axis("off")

# Show the completed figure in the notebook.
plt.show()


In [ ]:
# Inspect the image and its pixel values
# image.shape gives the dimensions of the image array:
# number of rows (height) and number of columns (width).
print("Image shape (rows, columns):", image.shape)

# image.min() finds the smallest pixel-intensity value.
# This corresponds to the darkest pixel in the image.
print("Minimum pixel value:", image.min())

# image.max() finds the largest pixel-intensity value.
# This corresponds to the brightest pixel in the image.
print("Maximum pixel value:", image.max())

# image.mean() calculates the average pixel intensity over the entire image.
# This gives a simple measure of the overall image brightness.
print("Mean pixel value:", image.mean())


In [ ]:
# Choose a threshold and segment the colony
# Set the threshold intensity value.
# Pixels brighter than 100 will be treated as colony pixels.
threshold = 100

# Compare every pixel value in 'image' with the threshold.
#
# For each pixel:
#   pixel value > 100  -> True
#   pixel value <= 100 -> False
#
# The result is a Boolean (True/False) array called image_threshold.
# It has the same dimensions as the original image.
image_threshold = image > threshold

# Create a new 6 x 6 inch figure.
plt.figure(figsize=(6, 6))

# Display the Boolean thresholded image.
# With the grayscale color map:
#   True pixels appear white.
#   False pixels appear black.
plt.imshow(image_threshold, cmap="gray")

# Add a title showing the threshold value used.
plt.title(f"Thresholded Image - Threshold = {threshold}")

# Hide the x- and y-axis ticks.
plt.axis("off")

# Display the thresholded image.
plt.show()


In [ ]:
# Compare the original and thresholded images
# plt.subplots(1, 2, ...) creates one figure with:
#   1 row
#   2 columns
# so that two images can be shown side-by-side.
#
# fig refers to the complete figure.
# axes contains the two individual plotting areas.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- LEFT PANEL ----

# axes[0] means the first plotting panel.
# Display the original grayscale microscope image there.
axes[0].imshow(image, cmap="gray")

# Add a title to the first panel.
axes[0].set_title("Original Image")

# Hide the axis ticks and numbers in the first panel.
axes[0].axis("off")

# ---- RIGHT PANEL ----

# axes[1] means the second plotting panel.
# Display the thresholded Boolean image there.
axes[1].imshow(image_threshold, cmap="gray")

# Add a title to the second panel.
# The f-string inserts the current value of 'threshold' into the title.
axes[1].set_title(f"Thresholded Image (threshold = {threshold})")

# Hide the axis ticks and numbers in the second panel.
axes[1].axis("off")

# tight_layout() automatically adjusts spacing so the two panels
# and their titles fit neatly without overlapping.
plt.tight_layout()

# Display the final side-by-side comparison.
plt.show()


In [ ]:
# np.sum() adds all values in the Boolean array.
#
# Because True = 1 and False = 0, this gives the number of pixels
# classified as part of the colony.

area_frame_26 = np.sum(image_threshold)

print("Colony area in frame 26 =", area_frame_26, "pixels")

In [ ]:
# ---------------------------------------------------------
# Measure E. coli colony area for all image frames
# ---------------------------------------------------------

# Create an empty list to store the colony area from each image.
# One area value will be added for every image frame.
area_values = []

# Create an empty list to store the corresponding time values.
time_values = []

# The images were taken every 5 minutes.
time_interval = 5


# Loop through all 37 image frames.
# range(37) gives the numbers:
# 0, 1, 2, ..., 36
for frame in range(37):

    # Build the file name for the current image.
    #
    # The image files are named using two digits:
    # frame 0  -> colony_growth_00.tif
    # frame 1  -> colony_growth_01.tif
    # frame 2  -> colony_growth_02.tif
    # ...
    # frame 26 -> colony_growth_26.tif
    #
    # :02d tells Python to write the number using two digits.
    image_path = DATA_DIR / f"colony_growth_{frame:02d}.tif"


    # Open the current TIFF image.
    pil_image = Image.open(image_path)


    # Convert the image into a NumPy array.
    # Each element of this array represents the brightness
    # value of one pixel in the image.
    image = np.array(pil_image)


    # Apply the same threshold that was used in class.
    #
    # Every pixel is compared with the value 100:
    #
    # pixel value > 100  -> True
    # pixel value <= 100 -> False
    #
    # The True pixels are treated as part of the bacterial colony.
    image_threshold = image > 100


    # Calculate the colony area.
    #
    # In Python:
    # True  = 1
    # False = 0
    #
    # Therefore, np.sum() counts all the True pixels.
    # This gives the colony area in units of pixels.
    colony_area = np.sum(image_threshold)


    # Add the colony area from this frame to the area list.
    area_values.append(colony_area)


    # Calculate the time corresponding to the current image.
    #
    # Since images are taken every 5 minutes:
    #
    # frame 0 -> 0 min
    # frame 1 -> 5 min
    # frame 2 -> 10 min
    # ...
    current_time = frame * time_interval


    # Add the calculated time to the time list.
    time_values.append(current_time)



# ---------------------------------------------------------
# Plot Colony Area vs Time
# ---------------------------------------------------------

# Create a new figure.
plt.figure(figsize=(8, 5))


# Plot colony area against time.
#
# time_values are plotted on the x-axis.
# area_values are plotted on the y-axis.
#
# We use a scatter plot so that each image frame
# appears as one individual data point.
plt.scatter(time_values, area_values)


# Label the x-axis.
plt.xlabel("Time (min)")


# Label the y-axis.
# The colony area is measured by counting pixels.
plt.ylabel("Area (pixels)")


# Add a title to the graph.
plt.title("E. coli Colony Area vs Time")


# Display the final graph.
plt.show()